# 02 — Exploratory data analysis

This notebook uses reusable data adapters and computes all measurements from the local MC-OCR files at run time. Analysis and galleries are limited to the KIE training subset. The 391-image validation subset is shown only as an inventory count; it is not opened or used to tune analysis choices.

Image sharpness, brightness, contrast, and text skew are exploratory heuristics, not dataset-provided quality labels.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from docucheck.data.eda import collect_field_geometry, collect_image_features, select_gallery_examples
from docucheck.data.loader import audit_dataset, load_kie_records, resolve_dataset_root
from docucheck.visualization.receipt import plot_receipt_record

SEED = 42
DATASET_ROOT = resolve_dataset_root(os.environ.get("MC_OCR_ROOT"))
audit = audit_dataset(DATASET_ROOT)
records = load_kie_records(DATASET_ROOT)
image_features = collect_image_features(records, DATASET_ROOT)
field_geometry = collect_field_geometry(records, image_features)
print(f"KIE train records: {len(records)} | analyzed images: {len(image_features)}")

## Available splits and image dimensions

The local KIE export has a labeled training set and a validation image folder whose sample CSV does not provide KIE label/polygon columns. We compare counts for inventory and use the training images for all plots below.

In [ ]:
split_counts = pd.Series({
    "KIE train (analyzed)": audit["image_inventory"]["train_image_count"],
    "validation (inventory only)": audit["image_inventory"]["validation_image_count"],
})
display(split_counts.rename_axis("subset").to_frame("images"))
valid_images = image_features[image_features.image_valid].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=valid_images, x="width", bins=20, ax=axes[0], color="#255f85")
axes[0].set_title("Receipt widths")
sns.histplot(data=valid_images, x="height", bins=20, ax=axes[1], color="#d17b49")
axes[1].set_title("Receipt heights")
for ax in axes:
    ax.set_xlabel("Pixels")
    ax.set_ylabel("Images")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=valid_images, x="width", y="height", hue="orientation", alpha=.7, ax=ax,
                palette={"portrait_or_square":"#255f85", "landscape":"#e4572e"})
ax.set_title("Resolution and orientation in KIE train")
plt.tight_layout()
plt.show()

## KIE label coverage and receipt annotation density

A missing labeled region means **no annotation with that source label was observed**. It does not prove the field is absent from the receipt. Source labels and the candidate canonical mapping are kept visible so rare/unmapped labels are not hidden.

In [ ]:
target_fields = list(audit["annotations"]["field_mapping"].values())
coverage = pd.DataFrame(index=[record["image_id"] for record in records])
for field in target_fields:
    coverage[field] = 0
for record in records:
    for region in record["regions"]:
        if region["canonical_field"]:
            coverage.loc[record["image_id"], region["canonical_field"]] += 1
coverage_summary = pd.DataFrame({
    "images_with_label": (coverage > 0).sum(),
    "images_without_label_annotation": (coverage == 0).sum(),
    "region_count": coverage.sum(),
})
display(coverage_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
(coverage > 0).sum().sort_values().plot(kind="barh", ax=axes[0], color="#3f8f78")
axes[0].set(xlabel="Images with label", title="KIE field annotation coverage")
region_counts = pd.Series([len(record["regions"]) for record in records], name="KIE regions")
sns.histplot(region_counts, discrete=True, ax=axes[1], color="#d17b49")
axes[1].set_title("KIE regions per receipt")
plt.tight_layout()
plt.show()
print("Observed raw labels:", audit["annotations"]["label_region_counts"])

## Full-page transcription and crop recognition distributions

The full-page sidecars provide text and quadrilateral geometry for a subset of training receipts. The separate recognition task provides word/line crop pairs. Transcript text remains in memory only; plots use lengths and region counts.

In [ ]:
ocr_stats = audit["ocr_exports"]
print("Full-page OCR sidecar coverage and parse status:", {
    key: ocr_stats[key] for key in ["full_page_sidecar_images", "train_images_without_full_page_sidecar",
                                    "full_page_text_regions", "full_page_malformed_rows"]
})
from docucheck.data.loader import load_full_page_text_records, load_text_recognition_pairs
page_gt = load_full_page_text_records(DATASET_ROOT)
fullpage_region_counts = pd.Series([len(record["regions"]) for record in page_gt], name="text regions")
recognition_lengths = [
    len(pair["text_raw"]) for split in ("train", "val")
    for pair in load_text_recognition_pairs(split, DATASET_ROOT) if pair["text_raw"] is not None
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(fullpage_region_counts, bins=25, ax=axes[0], color="#255f85")
axes[0].set(title="Full-page OCR regions per receipt", xlabel="Regions", ylabel="Receipts")
sns.histplot(recognition_lengths, bins=30, ax=axes[1], color="#d17b49")
axes[1].set(title="Crop transcription lengths", xlabel="Unicode code points", ylabel="Text crops")
plt.tight_layout()
plt.show()
print("Recognition task split counts:", ocr_stats["recognition_pairs"],
      "| transcription codepoint range:", ocr_stats["transcription_length_codepoints"])

## Relative spatial distribution of KIE fields

Centers are normalized by each receipt's width and height. The box-area measure uses axis-aligned bounds around each polygon and is a comparative approximation, not polygon area.

In [ ]:
field_geometry = field_geometry[field_geometry.canonical_field.notna()].copy()
counts = field_geometry.groupby("canonical_field").size().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for field, group in field_geometry.groupby("canonical_field"):
    sns.kdeplot(data=group, x="center_x_normalized", y="center_y_normalized", levels=5,
                thresh=.08, ax=axes[0], label=f"{field} (n={len(group)})")
axes[0].invert_yaxis()
axes[0].set(xlim=(0, 1), ylim=(1, 0), title="Normalized field-region centers", xlabel="x / width", ylabel="y / height")
axes[0].legend(fontsize=8)
sns.boxplot(data=field_geometry, x="canonical_field", y="bbox_area_fraction", ax=axes[1], color="#8fc5b3")
axes[1].set_yscale("log")
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Relative axis-aligned box area", xlabel="KIE field", ylabel="Fraction of receipt area (log)")
plt.tight_layout()
plt.show()
display(counts.rename("regions").to_frame())

## Image-quality heuristics and correlations

Brightness is grayscale mean intensity, contrast is grayscale standard deviation, and sharpness is Laplacian variance. Higher sharpness often indicates more high-frequency detail but can also reflect texture/noise. These are not official MC-OCR quality scores.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column, color, title in [
    (axes[0], "brightness_mean", "#d17b49", "Mean brightness"),
    (axes[1], "contrast_std", "#3f8f78", "Contrast heuristic"),
    (axes[2], "laplacian_variance", "#255f85", "Sharpness heuristic"),
]:
    sns.histplot(data=valid_images, x=column, bins=25, ax=ax, color=color)
    ax.set_title(title)
    ax.set_ylabel("Images")
plt.tight_layout()
plt.show()

correlations = valid_images[["brightness_mean", "contrast_std", "laplacian_variance", "ocr_region_count", "ocr_transcript_codepoints"]].corr(method="spearman")
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", center=0, ax=plt.subplots(figsize=(8, 6))[1])
plt.title("Spearman correlations (exploratory; n is limited to matched source records)")
plt.tight_layout()
plt.show()
print("Matched full-page text sidecars:", int((valid_images.ocr_region_count > 0).sum()), "of", len(valid_images))

## Local qualitative gallery — train only

Examples are selected by deterministic feature rankings from the training subset: sharpness extremes, text-edge tilt, region density, and longest annotated region. The selection labels describe the heuristic used; inspect the local images before drawing conclusions.

In [ ]:
gallery = select_gallery_examples(records, valid_images)
fig, axes = plt.subplots(len(gallery), 1, figsize=(10, 4.5 * len(gallery)))
if len(gallery) == 1:
    axes = [axes]
for (reason, record), ax in zip(gallery.items(), axes):
    plot_receipt_record(record, ax=ax, show_text=False)
    ax.set_title(reason)
plt.tight_layout()
plt.show()

feature_summary = valid_images[["brightness_mean", "contrast_std", "laplacian_variance",
                                "median_abs_text_edge_angle_degrees", "ocr_region_count"]].describe(percentiles=[.1,.5,.9]).T
display(feature_summary)

## EDA Findings

1. KIE field labels do not cover every image: observed field coverage ranges from **1,071 to 1,088 of 1,155 images**. Treat no-label cases as unannotated for that field, not proof that the receipt omits it.
2. Full-page transcription sidecars cover **1,151 of 1,155 training images**, with 40,868 regions. The four unmatched images need explicit handling in preparation; the sidecars provide broader OCR supervision than the KIE field annotations.
3. The crop recognition export has **5,285 train and 1,300 validation crop/transcript pairs**. This supports a separate crop-domain recognition check, not receipt-level field extraction.
4. Receipts vary substantially in size, brightness, sharpness, and text orientation. Those image metrics are heuristics and should be used to stratify error review, not as ground-truth quality labels.

**Follow-up:** preserve the official KIE and OCR task provenance, create a deterministic internal development split for the full-page train annotations, and report KIE missingness separately from OCR transcript availability.